# causal inference intro

treatment effects, simpson's paradox, propensity scores. dowhy preview.


In [1]:
import numpy as np
import pandas as pd

# synthetic confounded data
np.random.seed(0)
n = 5000
age = np.random.randint(20, 70, n)
smoke = (np.random.rand(n) < (age / 120)).astype(int)
outcome = 0.4 * smoke - 0.01 * age + np.random.normal(0, 1, n)
df = pd.DataFrame({'age': age, 'smoke': smoke, 'y': outcome})
df.head()


In [2]:
# naive estimate
naive = df[df.smoke==1]['y'].mean() - df[df.smoke==0]['y'].mean()
print('naive:', naive)


In [3]:
# stratify by age bucket
df['agebkt'] = pd.cut(df.age, bins=[20,30,40,50,60,70])
strat = df.groupby('agebkt').apply(lambda g: g[g.smoke==1].y.mean() - g[g.smoke==0].y.mean())
print(strat)


In [4]:
# weighted ate
weights = df.groupby('agebkt').size() / len(df)
ate = (strat * weights).sum()
print('stratified ate:', ate)


In [5]:
# propensity score with logistic regression
from sklearn.linear_model import LogisticRegression
p = LogisticRegression().fit(df[['age']], df['smoke']).predict_proba(df[['age']])[:,1]
df['ps'] = p


In [6]:
# IPW estimator
ipw = ((df.smoke * df.y / df.ps) - ((1 - df.smoke) * df.y / (1 - df.ps))).mean()
print('ipw ate:', ipw)


### naive estimate is biased by age confounding. ipw recovers true effect more closely.


### todo: try dowhy formal api on a real dataset.


In [ ]:
# todo: try lr=3e-4

minor tweak.

minor tweak.